# 09.05 · Agent frameworks compared

**Chapter:** Tool Use & Agents · **Milestone:** M4

## Goals
- [ ] Build the **same** agent (search the corpus + calculator) in four frameworks, all on the same model
- [ ] Know the rest of the landscape and pick by need, not hype

## Resources
- [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) · [PydanticAI](https://ai.pydantic.dev/) — free
- [LangChain agents / LangGraph](https://docs.langchain.com/) · [LangChain Academy](https://academy.langchain.com/) — free
- [smolagents](https://huggingface.co/docs/smolagents/) · [Hugging Face Agents Course](https://huggingface.co/learn/agents-course) — free
- [Claude Agent SDK](https://docs.claude.com/en/api/agent-sdk/overview) · [Google ADK](https://google.github.io/adk-docs/) · [CrewAI](https://docs.crewai.com/) · [Microsoft Agent Framework](https://learn.microsoft.com/en-us/agent-framework/) — free

| Framework | Style | Choose it for |
|---|---|---|
| **OpenAI Agents SDK** | small: agents, tools, handoffs, guardrails, tracing | multi-agent handoffs with little code; works with any OpenAI-compatible model |
| **PydanticAI** | typed, Pythonic, dependency injection | type-safe outputs, testability, FastAPI-style DX |
| **LangChain `create_agent` / LangGraph** | graph/state machine, checkpoints, interrupts | complex stateful flows, human-in-the-loop, durable execution |
| **smolagents** | minimal; **CodeAgent** writes Python instead of JSON calls | research, code-acting agents, HF ecosystem |
| **Claude Agent SDK** | the Claude Code harness as a library (files, bash, MCP, subagents) | coding/ops agents on Claude |
| **Google ADK** | multi-agent, Gemini/Vertex-first, eval tooling | Google Cloud shops |
| **CrewAI** | role-based "crews" | quick role-play multi-agent prototypes |
| **Microsoft Agent Framework** (AutoGen + Semantic Kernel successor) | enterprise, .NET + Python | Azure / Microsoft stacks |
| **Plain loop** (09.02) | ~60 lines | full control, simplest debugging |

In [ ]:
import ast
import operator

from openai import AsyncOpenAI

from aieng import rag
from aieng.llm import client

c, MODEL = client()
BASE_URL, API_KEY = str(c.base_url), c.api_key
LOCAL = c.base_url.host == "localhost"
NO_THINK = {"reasoning_effort": "none"} if LOCAL else {}
TASK = "In what year was Abraham Lincoln born, and how many years ago was that from 2026? Use the tools."
SYSTEM = "Use search_docs for facts and calculator for arithmetic. Answer in one sentence."

## Shared tools (plain functions)
Typed signatures + docstrings: every framework turns these into tool schemas automatically.

In [ ]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression like '2026 - 1809'. Supports + - * / ** and parentheses."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.operand))
        raise ValueError("only arithmetic is allowed")
    try:
        return str(ev(ast.parse(expression, mode="eval").body))  # safe: never eval() model output
    except (ValueError, SyntaxError, ZeroDivisionError) as e:  # return errors to the model (09.01); some frameworks crash on raised exceptions
        return f"Error: {e}. Pass a plain arithmetic expression like '2026 - 1809'."


def search_docs(query: str) -> str:
    """Search an encyclopedia for facts. Returns the 3 most relevant passages."""
    return "\n\n".join(f"[{pid}] {text}" for pid, _, text in rag.search(query, k=3))


print(calculator("2026 - 1809"))

## 1. OpenAI Agents SDK
Works with non-OpenAI models through `OpenAIChatCompletionsModel` + any OpenAI-compatible client.

In [ ]:
from agents import Agent, ModelSettings, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled

set_tracing_disabled(True)  # tracing uploads to OpenAI by default; off unless you have an OpenAI key
oa_agent = Agent(
    name="researcher", instructions=SYSTEM, tools=[function_tool(search_docs), function_tool(calculator)],
    model=OpenAIChatCompletionsModel(model=MODEL, openai_client=AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)),
    model_settings=ModelSettings(temperature=0, extra_body=NO_THINK),
)
result = await Runner.run(oa_agent, TASK)
print(result.final_output)
print("tool calls:", [item.raw_item.name for item in result.new_items if item.type == "tool_call_item"])

## 2. PydanticAI
Typed agents; `output_type` makes the final answer a validated Pydantic object.

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent as PydanticAgent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


class Answer(BaseModel):
    birth_year: int
    years_ago: int


pa_agent = PydanticAgent(OpenAIChatModel(MODEL, provider=OpenAIProvider(base_url=BASE_URL, api_key=API_KEY)),
                         system_prompt=SYSTEM, tools=[search_docs, calculator], output_type=Answer,
                         model_settings={"temperature": 0, "extra_body": NO_THINK})
try:
    r = await pa_agent.run(TASK)
    print(r.output, "| usage:", r.usage())
except Exception as e:  # validation failed even after PydanticAI re-asked the model with the error
    print(f"PydanticAI gave up: {type(e).__name__}: {str(e)[:150]}")
    print("Small models often break typed outputs; that's a feature: you get an exception, not silently malformed data.")

## 3. LangChain `create_agent` (built on LangGraph)

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

lc_agent = create_agent(ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY, temperature=0, **NO_THINK),
                        tools=[search_docs, calculator], system_prompt=SYSTEM)
out = await lc_agent.ainvoke({"messages": [{"role": "user", "content": TASK}]})
print(out["messages"][-1].content)
print("tool calls:", [tc["name"] for m in out["messages"] for tc in getattr(m, "tool_calls", [])])

## 4. smolagents — `ToolCallingAgent` and `CodeAgent`
`CodeAgent` writes Python that calls tools (often fewer steps: loops, variables, arithmetic in code). It runs that code in a restricted
local interpreter — use a remote sandbox (E2B/Modal/Docker) for anything untrusted (09.08).

In [ ]:
from smolagents import CodeAgent, OpenAIServerModel, ToolCallingAgent, tool

smol_model = OpenAIServerModel(model_id=MODEL, api_base=BASE_URL, api_key=API_KEY, temperature=0, **NO_THINK)


@tool
def smol_search(query: str) -> str:
    """Search an encyclopedia for facts. Returns the 3 most relevant passages.

    Args:
        query: what to search for
    """
    return search_docs(query)


tc_agent = ToolCallingAgent(tools=[smol_search], model=smol_model, max_steps=5, verbosity_level=0)
print("ToolCallingAgent:", tc_agent.run(TASK))
code_agent = CodeAgent(tools=[smol_search], model=smol_model, max_steps=5, verbosity_level=0)
try:
    print("CodeAgent:", code_agent.run(TASK))
except Exception as e:  # small models often write invalid code for CodeAgent's format
    print("CodeAgent failed:", type(e).__name__, str(e)[:200])

## 5. The others, for reference (not installed)
```python
# Claude Agent SDK (needs Claude Code + ANTHROPIC_API_KEY): `uv add claude-agent-sdk`
from claude_agent_sdk import query, ClaudeAgentOptions
async for msg in query(prompt=TASK, options=ClaudeAgentOptions(mcp_servers={"wiki": {"command": "uv", "args": [...]}})):
    print(msg)

# Google ADK: `uv add google-adk`
from google.adk.agents import Agent
root_agent = Agent(name="researcher", model="gemini-flash-latest", instruction=SYSTEM, tools=[search_docs, calculator])

# CrewAI: `uv add crewai`
from crewai import Agent, Task, Crew
researcher = Agent(role="Researcher", goal="Find facts", backstory="...", tools=[...], llm="ollama/qwen3:1.7b")
Crew(agents=[researcher], tasks=[Task(description=TASK, expected_output="one sentence", agent=researcher)]).kickoff()
```

## 6. How to choose
1. Start with a plain loop or the thinnest SDK (OpenAI Agents SDK / PydanticAI). Upgrade when you *need* something specific.
2. Need durable state, checkpoints, human approval, complex branching → **LangGraph**.
3. Need typed outputs and testability → **PydanticAI**.
4. Building on a specific vendor's platform (Claude, Gemini/Vertex, Azure) → their SDK gets new features first.
5. Whatever you pick: tracing (chapter 11) and evals (chapter 10) matter more than the framework.

## Exercises
1. Count lines of code for each version and list what each framework gave you for free (retries, tracing, typed output, streaming…).
2. Add streaming output to the OpenAI Agents SDK version (`Runner.run_streamed`) and the PydanticAI version (`agent.run_stream`).
3. Install one of the reference frameworks and port the agent. Write a 5-line verdict.

In [ ]:
# your code here